# Übungsblatt 3 in "Mechanistische Effektmodellierung" 2026: Dynamic Energy Budget Modelle

### Hinweis:
Entpacken Sie die zip-Datei, die zu diesem Übungsblatt gehört, in dem Ordner, in dem sich dieses Notebook befindet. Sie enthält den Ordner `data` und die Datei `Uebungsblatt3_MEM26_Hilfsfunktionen.py`.

<i>
Hinweis:

Sie können mempyDEB mit dem folgenden command in Ihr conda environment installieren:
</i>
<code>
pip install git+https://github.com/simonhansul/mempyDEB.git
</code>

## Aufgabe 1: DEBkiss-Grundlagen

Im Folgenden sind noch einmal die Gleichungen der DEB-Modellvariante gegeben, die Sie schon aus der Vorlesung kennen.

$$
\dot{I} = \begin{cases}
    f(X)\, \{\dot{I}_{max}\}\, S^{2/3}\ \ \text{mit}\ \ f(X) = \frac{[X]}{[X] + K_X} & \text{für}\ \ X^{emb} = 0 \\
    \{\dot{I}_{max}^{emb}\}\, S^{2/3} & \text{für}\ \ X^{emb} > 0
\end{cases}
$$

$$
\dot{X}^{emb} = \begin{cases}
    -\dot{I} & \text{für}\ \ X^{emb} > 0 \\
    0 & \text{für}\ \ X^{emb} = 0
\end{cases}
$$

$$
\dot{X} = \begin{cases}
    \dot{X}_{in} - \dot{I} & \text{für}\ \ X^{emb} = 0 \\
    0 & \text{für}\ \ X^{emb} > 0
\end{cases}
$$

$$
\dot{A} = \eta_{IA}\, \dot{I}
$$

$$
\dot{M} = k_M\, S
$$

$$
\dot{S} = \begin{cases}
    \eta_{AS} \left(\kappa \dot{A} - \dot{M}\right) & \text{für}\ \ \kappa \dot{A} \geq \dot{M} \\
    -\left(\dot{M} / \eta_{SA} - \kappa \dot{A}\right) & \text{für}\ \ \kappa \dot{A} < \dot{M}
\end{cases}
$$

$$
\dot{R} = \begin{cases}
    \eta_{AR}\, (1 - \kappa)\, \dot{A} & \text{für}\ \ S \geq S_p \\
    0 & \text{für}\ \ S < S_p
\end{cases}
$$

### Symbolerklärung (Zustandsgrößen / Raten)

| Größe | Dimension | Variablenname in Python | Interpretation |
|------|---------|-------------------------|----------------|
| $\dot{I}$ | $m\ t^{-1}$ | `Idot` | Nahrungs-Aufnahmerate |
| $\dot{A}$ | $m\ t^{-1}$ | `Adot` | Assimilierungsrate |
| $\dot{M}$ | $m\ t^{-1}$ | `Mdot` | Somatische Instandhaltungsrate |
| $\dot{S}$ | $m\ t^{-1}$ | `Sdot` | Strukturelle Wachstumsrate |
| $\dot{R}$ | $m\ t^{-1}$ | `Rdot` | Zunahme des Reproduktionspuffers |

### Symbolerklärung (Parameter)

| Parameter | Dimension | Variablenname in Python | Interpretation |
|-----------|---------|---------------|----------------|
| $X_{int}^{emb}$ | $m$ | `X_emb_int` | Anfängliche Masse des Vitellus |
| $K_X$ | $m\ V^{-1}$ | `K_X` | Halbsättigungskonstante für Nahrungsaufnahmerate |
| $V$ | $L$ | `V_patch` | Volumen des umgebenden Mediums (z.B. Reagenzglas oder Aquarium im Labor, Habitats-Patch) |
| $\dot{X}_{in}$ | $m\ t^{-1}$ | `Xdot_in` | Nahrungseintragsrate |
| $\{\dot{I}_{max}\}$ | $m\ m^{-2/3}\ t^{-1}$ | `Idot_max_rel` | Maximale relative Nahrungsaufnahmerate für Juvenile und Adulte |
| $\{\dot{I}_{max}^{emb}\}$ | $m\ m^{-2/3}\ t^{-1}$ | `Idot_max_rel_emb` | Maximale relative Nahrungsaufnahmerate für Embryonen |
| $\kappa$ | $-$ | `kappa` | somatischer Allokationsanteil |
| $\eta_{IA}$ | $-$ | `eta_IA_0` | Assimilierungseffizienz |
| $\eta_{AS}$ | $-$ | `eta_AS_0` | Wachstumseffizienz |
| $\eta_{AR}$ | $-$ | `eta_AR_0` | Reproduktionseffizienz |
| $\eta_{SA}$ | $-$ | `eta_SA` | Effizienz der Umsetzung von Struktur in Assimilate |
| $k_M$ | $t^{-1}$ | `k_M_0` | somatische Instandhaltungsratenkonstante |
| $S_p$ | $m$ | `S_p` | Maturitäts-Schwellwert für Pubertät |

Für eine fiktive Wasserfloh-Art (*Daphnia* spec.) sind folgende Werte bekannt:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from mempyDEB.DEBODE.simulators import *
from mempyDEB.DEBODE.defaultparams import *

S_max_obs_dspec = 13.5 # beobachtete maximale strukturelle Masse (mugC)
Idot_max_obs = 30      # Nahrungsaufnahmerate bei maximaler Größe (mugC d^-1)
Adot_max_obs = 10      # Assimilierungsrate bei maximaler Größe (mugC d^-1)

dspec = deepcopy(defaultparams_DEBBase.spc) # Standardparameter als Ausgangspunkt

### A1a
Leiten Sie aus den Modellgleichungen einen Ausdruck für die maximale strukturelle Masse $S_{max}$ her, die ein Individuum bei Nahrung im Überfluss erreicht. Geben Sie den Rechenweg als LaTeX-Gleichungen oder eingebettetes Foto in einer Markdown-Zelle an und berechnen Sie $S_{max}$ für die Standardparameter in `dspec`. Der Fall des negativen Wachstums kann ignoriert werden.

Antwort:

In [ ]:
# Lösung

### A1b
Berechnen Sie aus den gegebenen beobachteten Werten Schätzwerte für die größenspezifische maximale Nahrungsaufnahmerate $\{\dot{I}_{max}\}$, die Assimilierungseffizienz $\eta_{IA}$ und die somatische Instandhaltungsratenkonstante $k_M$. Nehmen Sie an, dass die Werte bei Nahrung im Überfluss gemessen wurden, und verwenden Sie für $\kappa$ den Standardwert aus `dspec`. Die Rechenwege ergeben sich aus den Modellgleichungen; für $k_M$ benötigen Sie Ihr Ergebnis aus A1a. Geben Sie die Rechenwege an und speichern Sie die Ergebnisse im Dictionary `dspec` (steht für *Daphnia spec*, d.h. eine nicht näher bestimmte Art des Genus *Daphnia*). Nutzen Sie dafür die vorbereitete Code-Zelle unten, in der die Schlüssel schon vorgegeben sind.

In [ ]:
dspec['Idot_max_rel'] = #todo
dspec['eta_IA_0'] = #todo
dspec['k_M_0'] = #todo
dspec

Antwort:

### A1c
Simulieren Sie das Modell mit den berechneten Parameterwerten über 56 Tage und plotten Sie die strukturelle Masse und den Reproduktionspuffer über der Zeit. Kennzeichnen Sie $S_{max}$ im Plot. Erreicht die simulierte Struktur den beobachteten Wert? Begründen Sie, wie sich aus den Modellgleichungen die asymptotische Annäherung ergibt. Die Nahrungsverfügbarkeit ist hier so hoch, dass $f \approx 1$ ist und die Nahrung somit nicht limitierend ist.

<i>Hinweis: Ein Parameter-Objekt erzeugen Sie mit `p = Params(glb = deepcopy(defaultparams_DEBBase.glb), spc = deepcopy(dspec))`. Globale Größen wie die Simulationsdauer (`p.glb['tspan'] = (0, 56)`) oder die Nahrungszufuhr (`p.glb['Xdot_in']`) stehen in `p.glb`, die artspezifischen Parameter in `p.spc`. Mit `sim = simulate_DEBBase(p)` erhalten Sie einen DataFrame mit der Zeit in Spalte `t` und den Zustandsgrößen (z.B. `S` und `R`). Die Zeitspalte `t` ist um das Anfangsalter `p.glb['a_int']` (2 Tage, inklusive Embryonalentwicklung) verschoben: Mit `tspan = (0, 56)` läuft `t` von −2 bis 54.</i>

In [ ]:
p = Params(
    glb = deepcopy(defaultparams_DEBBase.glb), 
    spc = deepcopy(dspec)
)

In [ ]:
# Lösung

Antwort:

### A1d
Untersuchen Sie, wie sich $\kappa$ (`kappa`) und $\eta_{AS}$ (`eta_AS_0`) auf Wachstum und Reproduktion auswirken. Simulieren Sie das Modell mit den Standardparametern (`defaultparams_DEBBase.spc`, also nicht mit den Werten aus A1b) über 56 Tage, jeweils für $\kappa \in \{0.3, 0.6, 0.9\}$ und $\eta_{AS} \in \{0.3, 0.6, 0.9\}$ (der jeweils andere Parameter bleibt auf dem Standardwert), und plotten Sie $S$ und $R$ über der Zeit. Beschreiben Sie die Effekte und erklären Sie sie mithilfe Ihres Ausdrucks für $S_{max}$ aus A1a und der Modellgleichungen.

In [ ]:
# Lösung

Antwort:

Bisher haben wir eine fiktive *Daphnia*-Art betrachtet. Für die Kalibrierung in Aufgabe 2 benötigen wir Startwerte für *Daphnia magna*, die deutlich größer wird: Für *D. magna* wurde eine maximale strukturelle Masse von $388.4\ \mu g\,C$ beobachtet.

In der DEB-Theorie geht man davon aus, dass sich verwandte Arten vor allem in ihrer maximalen Größe unterscheiden. Parameter lassen sich dann mit einem Zoom-Faktor von einer Art auf eine andere übertragen: Einige Parameter skalieren mit der Körpergröße (z.B. die größenspezifische maximale Nahrungsaufnahmerate oder die Größe bei der Pubertät), andere bleiben gleich (z.B. die Effizienzen). So erhält man ohne eigene Messungen plausible Startwerte, die anschließend an Daten kalibriert werden.

In [ ]:
S_max_obs_dmagna = 388.4 # mugC

### A1e
Wir definieren den Zoom-Faktor $Z$ hier über das Verhältnis der maximalen strukturellen Massen: $Z = S_{max}^{magna} / S_{max}^{spec}$. Berechnen Sie $Z$ und passen Sie $\{\dot{I}_{max}\}$ so an, dass $S_{max}$ dem Wert von *D. magna* entspricht. Speichern Sie die skalierten Parameter in `dmagna.spc` (vorbereitete Zelle unten). `dmagna.spc` ist eine Kopie von `dspec`, enthält also Ihre Werte aus A1b. Begründen Sie mithilfe Ihres Ergebnisses aus A1a, wie $Z$ dabei eingeht. Welche weiteren Parameter sollten mit der Körpergröße skalieren? Begründen Sie und skalieren Sie diese Parameter ebenfalls.

Simulieren Sie *D. magna* bei Nahrung im Überfluss (`Xdot_in` sehr groß, z.B. `1e10`) über 56 Tage und prüfen Sie, ob sich die strukturelle Masse $S_{max}$ annähert.

In [ ]:
dmagna = Params(
    glb = deepcopy(defaultparams_DEBBase.glb), 
    spc = deepcopy(dspec)
)
dmagna.glb['tspan'] = (0, 56)

In [ ]:
# Lösung

Antwort:

## Aufgabe 2: Kalibrierung des DEB-Modells an Kontrolldaten

In Aufgabe 1 haben wir Parameterwerte aus wenigen Beobachtungen und mit dem Zoom-Faktor abgeleitet. Das sind nur grobe Schätzungen. Jetzt kalibrieren wir das Modell an echten Messdaten mit dem Nelder-Mead-Algorithmus aus `scipy.optimize.minimize`. Der Algorithmus sucht die Parameterwerte, für die die Verlustfunktion am kleinsten ist. Sie misst die Abweichung zwischen Modell und Daten: Je kleiner ihr Wert, desto besser die Anpassung. Erst wenn das Modell die Kontrolle gut beschreibt, können wir in Aufgabe 3 die Effekte einer Chemikalie modellieren.

Wir verwenden Daten eines Experiments mit *D. magna*: strukturelle Masse und kumulative Reproduktion in der Kontrolle (ohne Chemikalie) über 21 Tage. Als Startwerte dienen die Parameter für *D. magna* aus A1e. Sie sind in `Uebungsblatt3_MEM26_Hilfsfunktionen.py` fest hinterlegt, damit Aufgabe 2 nicht von Ihrem Ergebnis aus Aufgabe 1 abhängt. In dieser Datei befinden sich auch die Funktionen zum Einlesen, Plotten und Kalibrieren.

In [ ]:
from Uebungsblatt3_MEM26_Hilfsfunktionen import load_control_data, plot_control_data, setup_modelfit, fit_model

data = load_control_data()
fig, ax = plot_control_data(data)

### A2a
Kalibrieren Sie zunächst nur die größenspezifische maximale Nahrungsaufnahmerate `Idot_max_rel` mit dem folgenden Code. Beschreiben Sie die Anpassungsgüte: Welche Aspekte der Daten (Wachstum, Beginn und Höhe der Reproduktion) bildet das Modell gut ab, welche nicht?

In [ ]:
fit_1 = fit_model()
fit_1.optimization_result

Antwort:

### A2b
Wählen Sie aus `kappa`, `eta_AS_0` und `S_p` einen weiteren Parameter, von dem Sie denken, dass er die Anpassungsgüte am stärksten verbessern wird, und kalibrieren Sie ihn zusätzlich zu `Idot_max_rel`. Begründen Sie Ihre Wahl anhand der Modellgleichungen und Ihrer Beobachtungen aus A2a. Vergleichen Sie die Anpassungsgüte (Plots und Wert der Verlustfunktion) mit A2a. Hat sich die Anpassung verbessert? Warum bzw. warum nicht?

<i>Hinweis: Schreiben Sie dafür im Notebook eine eigene Funktion nach dem Vorbild von `fit_model()` aus `Uebungsblatt3_MEM26_Hilfsfunktionen.py`. `setup_modelfit()` gibt ein `ModelFit`-Objekt zurück, das Daten, Simulator, Verlustfunktion, Parameter und Optimierer enthält. Ändern müssen Sie nur, welche Parameter kalibriert werden: Diese stehen mit ihren Startwerten im Attribut `intguess` des `ModelFit`-Objekts. Alle Parameter, die nicht in `intguess` stehen, bleiben auf ihren Standardwerten. Den Wert der Verlustfunktion finden Sie nach der Kalibrierung im Attribut `optimization_result` (`.fun`).</i>

In [ ]:
# Lösung

Antwort:

### A2c
Kalibrieren Sie nun alle vier Parameter gleichzeitig: `Idot_max_rel`, Ihre Wahl aus A2b und die beiden übrigen Parameter aus [`kappa`, `eta_AS_0`, `S_p`]. Vergleichen Sie die Anpassungsgüte (Plots und Wert der Verlustfunktion) mit A2b. Hat sich die Anpassung verbessert? Was ändert sich im Modellverhalten durch die angepassten Parameterwerte?

In [ ]:
# Lösung

Antwort:

### A2d
Mehr kalibrierte Parameter führen meist zu einer besseren Anpassung. Warum ist es trotzdem nicht immer sinnvoll, möglichst viele Parameter gleichzeitig zu kalibrieren? Beziehen Sie sich auf die Daten und Ergebnisse dieser Aufgabe.

Antwort:

## Aufgabe 3: DEB-TKTD-Modell für Azoxystrobin

Bisher haben wir nur Tiere ohne Chemikalie betrachtet. Jetzt erweitern wir das DEB-Modell um einen TKTD-Teil (DEB-TKTD, auch DEBtox genannt), um subletale Effekte auf Wachstum und Reproduktion zu beschreiben. Wie bei GUTS (Blatt 1) baut sich im Tier ein skalierter Schaden $D$ auf. Anders als bei GUTS führt der Schaden aber nicht zum Tod, sondern verändert einen Parameter des DEB-Modells. Welcher Prozess dabei betroffen ist, nennt man die physiologische Wirkungsweise (*physiological mode of action*, PMoA).

Wir betrachten alle Konzentrationen (0 bis 250 nmol/L) des Experiments mit *D. magna* und dem Fungizid Azoxystrobin. Die Daten sind als relative Antwort bezogen auf die Kontrolle dargestellt: $y_S = S/S_{Kontrolle}$ für die strukturelle Masse und $y_R = N/N_{Kontrolle}$ für die kumulative Anzahl der Nachkommen $N$. Solange in der Kontrolle noch keine Nachkommen vorhanden sind (in den Daten vor Tag 11), wird $y_R = 1$ gesetzt. Für die Konzentrationen 150 und 250 gibt es keine Wachstumsdaten, deshalb bleiben diese Wachstumsplots leer. Die DEB-Parameter sind auf Werte fixiert, die an die Kontrolldaten kalibriert wurden. Sie sind in `Uebungsblatt3_MEM26_Hilfsfunktionen.py` fest hinterlegt (`define_defaultparams_tox()`), damit Aufgabe 3 nicht von Ihrem Ergebnis aus Aufgabe 2 abhängt. Kalibriert werden nur die TKTD-Parameter $k_D$ (`kD_j`), $ED_{50}$ (`ED50_j`) und $\beta$ (`beta_j`).

**Erweiterung des Modells aus Aufgabe 1**

Schaden (Toxikokinetik):

$$
\dot{D} = k_D\,\frac{L_{max}}{L}\,\left(C_W - D\right) - D\,\frac{\dot{S}}{S} \qquad \text{mit}\ \ L = S^{1/3},\ \ L_{max} = S_{max}^{1/3}
$$

Der erste Term entspricht dem Schaden in GUTS-RED, ist aber größenabhängig: Kleine Tiere gleichen sich schneller an die äußere Konzentration $C_W$ an. Der zweite Term beschreibt die Verdünnung des Schadens durch Wachstum. Embryonen nehmen keine Chemikalie auf.

Wirkung des Schadens (Toxikodynamik) über die log-logistische Funktion

$$
y(D) = \frac{1}{1 + \left(D / ED_{50}\right)^{\beta}}
$$

Ohne Schaden ist $y = 1$, bei $D = ED_{50}$ ist $y = 0.5$; $\beta$ bestimmt, wie steil $y$ abfällt. Der Schaden wirkt auf genau einen Prozess (PMoA): `G` (Wachstum), `M` (Instandhaltung), `A` (Assimilation) oder `R` (Reproduktion).

### A3a
Welcher Parameter des DEB-Modells aus Aufgabe 1 wird bei jeder der vier PMoAs durch den Schaden verändert? Wird er mit zunehmendem Schaden kleiner oder größer? Für welche PMoAs kann man den Parameter deshalb einfach mit $y(D)$ multiplizieren, für welche nicht? Begründen Sie.

Antwort:

### A3b
Beschreiben Sie die beobachteten Effekte von Azoxystrobin auf Wachstum und Reproduktion. Welche PMoA erwarten Sie anhand der Daten? Begründen Sie mithilfe der Modellgleichungen aus Aufgabe 1 und Ihrer Zuordnung aus A3a, wie sich die vier PMoAs jeweils auf Wachstum und Reproduktion auswirken.

In [ ]:
from Uebungsblatt3_MEM26_Hilfsfunktionen import load_tox_data, plot_tox_data, fit_debtox

tox_data = load_tox_data()
fig, ax = plot_tox_data(tox_data)

Antwort:

### A3c
Kalibrieren Sie das DEB-TKTD-Modell für jede der vier PMoAs (z.B. `fit_debtox(tox_data, pmoa='G')`). Vergleichen Sie die Anpassungsgüte anhand der Plots und der Werte der Verlustfunktion. Welche PMoA beschreibt die Daten am besten? Deckt sich das mit Ihrer Erwartung aus A3b?

<i>Hinweis: Eine Kalibrierung dauert etwa eine halbe bis eine Minute, je nach Rechner auch länger.</i>

In [ ]:
# Lösung

Antwort:

### A3d
Interpretieren Sie die geschätzten Parameterwerte $k_D$, $ED_{50}$ und $\beta$ für die PMoA mit der besten Anpassung. Was sagen sie über die Geschwindigkeit und Stärke der Effekte aus?

Antwort: